# Допуск: Mem0: Building Production-Ready AI Agents with Scalable Long-Term Memory

**Статья:** P. Chhikara, D. Khant, S. Aryan, T. Singh, D. Yadav, 2025 —
<https://arxiv.org/abs/2504.19413>

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — названия функций заменены: определите, что чему соответствует, и
  ответьте на вопросы;
- **вопрос по коду** — в коде нет отклонений: ответьте на вопросы, опираясь на код и статью;
- **расхождение со статьёй** — код реализует не то, что описано в статье: найдите это место;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду: найдите
  его.

Для найденного отклонения напишите: где оно, что не так, какое место статьи (раздел /
формула / таблица / алгоритм / приложение / цитата) нарушено и как должно быть. Код писать
не нужно — достаточно объяснить словами. Запускать и отлаживать код тоже не нужно: все
задания решаются чтением кода и статьи.

Общие импорты:

In [ ]:
import json
from collections import Counter
from statistics import mean, stdev
import torch

Разговор `conv` — два собеседника (`conv.speakers`) и список сообщений `conv.messages`, у
каждого сообщения есть `time`, `speaker`, `text`. `llm(prompt)` — вызов языковой модели (в
статье GPT-4o-mini), `embed(texts)` — плотные эмбеддинги, тензор `(N, D)` с единичными
строками. База воспоминаний `db` хранит записи с полями `id`, `text`, `time`;
`db.search(text, top_k)` возвращает `top_k` самых похожих записей по косинусной близости,
по убыванию, с полем `score`. Функции, которые используются, но не показаны, считайте
корректными.

## Задание 1. Пять способов собрать контекст для ответа

**Тип задания:** что есть что — названия функций заменены, определите, что чему соответствует.

Каждая функция возвращает контекст, который вместе с вопросом подаётся модели, отвечающей
на вопрос LOCOMO (промпт — прил. A). Определите, какая функция соответствует подходам
**Full-context**, **RAG**, **OpenAI** (память ChatGPT), **Mem0** и **Mem0^g** (§3.3,
табл. 2). Ошибок в коде нет.

In [ ]:
def context_a(conv, question):
    return "\n".join(f"({m.time}) {m.speaker}: {m.text}" for m in conv.messages)

def context_b(conv, question, size=256, k=2):
    pieces = split_into_chunks(conv.messages, n_tokens=size)        # fixed-length chunks
    sims = embed([question]) @ embed(pieces).T                       # (1, N_chunks)
    return "\n\n".join(pieces[i] for i in sims[0].topk(k).indices)

def context_c(conv, question):
    memories = vendor_memories(conv)        # written once by the hosted assistant from the whole log
    return "\n".join(m.text for m in memories)

def context_d(conv, question, top_k):
    parts = []
    for spk in conv.speakers:
        mems = memory_db[spk].search(question, top_k)                # dense similarity search
        parts.append(f"Memories for user {spk}:\n" + "\n".join(f"{m.time}: {m.text}" for m in mems))
    return "\n\n".join(parts)

def context_e(conv, question, top_k):
    parts = []
    for spk in conv.speakers:
        mems = memory_db[spk].search(question, top_k)
        rels = graph_db[spk].search(question)                        # (source, relation, destination)
        parts.append(f"Memories for user {spk}:\n" + "\n".join(f"{m.time}: {m.text}" for m in mems)
                     + f"\n\nRelations for user {spk}:\n" + "\n".join(map(str, rels)))
    return "\n\n".join(parts)

**Вопросы.**
1. Какой функции соответствует каждое из значений столбца «chunk size / memory tokens»
   табл. 2: 26031, 256 (при K = 2), 4437, 1764, 3616? У каких двух функций в табл. 2 нет
   search latency и почему? Почему очень низкая total latency одной из них (p50 0.466 с) не
   значит, что этот подход дешёвый, и в чём авторы видят её «привилегированный» доступ к
   памяти (§3.3, §4.4)?
2. `context_a` даёт лучший Overall J (72.90), но во сколько раз её total p95 больше, чем у
   `context_d`? Проверьте по табл. 2 числа «91 %» (аннотация) и «92 % / 85 %» (§4.3). Что
   растёт с длиной разговора у `context_a` и что у `context_d`?
3. Почему в `context_b` берут только k ∈ {1, 2} (§3.3)? Какая конфигурация RAG лучшая по
   табл. 2, насколько от неё отстаёт `context_d` и как авторы объясняют, почему короткие
   факты лучше сырых кусков разговора (§4.3)?
4. По табл. 1 найдите категории вопросов, где `context_e` выигрывает у `context_d` по J, и
   где проигрывает. Согласуется ли это с тем, зачем графовая память вводится в §2? При этом
   Overall J в табл. 2 у Mem0^g выше (68.44 против 66.88). Посчитайте простое среднее
   четырёх значений J из табл. 1 для обеих систем и объясните, как получается Overall J.

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Фаза извлечения

**Тип задания:** вопрос по коду — в коде нет отклонений, ответьте на вопросы.

Фаза извлечения Mem0 (§2.1, рис. 2) для новой пары сообщений. `db.get_summary` возвращает
сводку разговора S из базы, `db.request_summary_refresh` ставит её пересчёт в очередь
отдельного процесса, `update_memory` — фаза обновления.

In [ ]:
EXTRACT = ("Conversation summary:\n{summary}\n\nRecent messages:\n{recent}\n\nNew messages:\n{pair}\n\n"
           "Extract salient facts stated in the NEW messages only. Make every fact self-contained: "
           "name the person, resolve pronouns and relative dates. Return a JSON list of strings.")

def fmt(messages):
    return "\n".join(f"({m.time}) {m.speaker}: {m.text}" for m in messages)

def extract_facts(llm, db, conv_id, messages, m=10):
    # messages: the conversation so far; messages[-2:] is the new pair (m_{t-1}, m_t)
    summary = db.get_summary(conv_id)                                # S
    recent = messages[-m - 1:-2]                                     # m_{t-m}, ..., m_{t-2}
    prompt = EXTRACT.format(summary=summary, recent=fmt(recent), pair=fmt(messages[-2:]))
    return json.loads(llm(prompt))                                   # Omega = [omega_1, ..., omega_n]

def ingest(llm, db, conv_id, messages):
    db.append_messages(conv_id, messages[-2:])
    db.request_summary_refresh(conv_id)                 # runs in the background, returns at once
    facts = extract_facts(llm, db, conv_id, messages)
    update_memory(db, facts)                            # update phase (not shown)
    return facts

**Вопросы.**
1. Зачем экстрактору и сводка S, и последние сообщения: что даёт каждая из частей (§2.1)?
   Пусть новая пара — «— So, did you go in the end? — Yes, yesterday, and it was so
   powerful!». Какой самодостаточный факт можно из неё получить и какие части промпта для
   этого нужны?
2. Факты извлекаются только из новой пары, хотя в промпте есть и более старые сообщения.
   Почему? Что стало бы с базой, если бы модель извлекала факты из всего, что видит в
   промпте? Как стоимость одного вызова `ingest` зависит от длины разговора?
3. Почему сводка пересчитывается в фоне, а не прямо перед `extract_facts` (§2.1)? Чем за
   это приходится платить и какая часть промпта это компенсирует?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Фаза обновления

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Фаза обновления Mem0 (§2.1, алгоритм 1, прил. B): факты из фазы извлечения сравниваются с
уже сохранёнными воспоминаниями, s = 10. Найдите одно отклонение от статьи.

In [ ]:
def update_memory(db, facts, s=10):
    for fact in facts:                                               # omega_i in Omega
        similar = db.search(fact, top_k=s)                           # s most similar memories
        best = similar[0] if similar else None
        if best is None or best.score < 0.7:
            op = "ADD"
        elif best.score < 0.9:
            op = "UPDATE"
        else:
            op = "NOOP"
        if op == "ADD":
            db.add(fact)                                             # new memory
        elif op == "UPDATE":
            db.replace(best.id, fact)                                # richer version of a memory
        elif op == "DELETE":
            db.delete(best.id)                                       # contradicted memory
        # NOOP: the fact is already stored

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Графовая память: конфликт отношений

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Добавление одного отношения в графовую память Mem0^g (§2.2, рис. 3). `graph.find_node`
ищет узел, эмбеддинг которого похож на эмбеддинг сущности не меньше порога t,
`resolver.obsolete` — LLM-модуль, который среди отношений-кандидатов находит устаревшие
из-за нового. Найдите одно отклонение от статьи.

In [ ]:
def add_relation(graph, triplet, resolver, t=0.8):
    # triplet = (source, relation, destination) from the relationship generator
    src, rel, dst = triplet
    u = graph.find_node(src, threshold=t) or graph.add_node(src)     # match by entity embedding
    v = graph.find_node(dst, threshold=t) or graph.add_node(dst)
    candidates = graph.edges_of(u)                                   # conflict detection
    for e in resolver.obsolete(candidates, triplet):                 # LLM update resolver
        graph.remove_edge(e)
    graph.add_edge(u, rel, v, created_at=now())

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Метрики качества ответа

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

Подсчёт метрик табл. 1 для одной категории вопросов (§3.1–3.2, прил. A). `bleu1` — BLEU-1
(униграммная точность со штрафом за краткость), `judge(q, gold, answer)` — LLM-судья с
промптом из прил. A: 1 за CORRECT, 0 за WRONG. Найдите один неверный комментарий.

In [ ]:
def token_f1(pred, gold):
    p, g = pred.lower().split(), gold.lower().split()
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(p), common / len(g)
    return 2 * precision * recall / (precision + recall)

def evaluate_category(items, judge, n_runs=10):
    # items: (question, gold, generated) for one question category; adversarial ones excluded
    f1 = 100 * mean(token_f1(a, g) for _, g, a in items)
    b1 = 100 * mean(bleu1(a, g) for _, g, a in items)
    # Lexical overlap already punishes factual errors: for the gold "Alice was born in March"
    # the answer "Alice is born in July" gets F1 = 0.2, so the judge is needed only to accept
    # long or paraphrased answers
    runs = [100 * mean(judge(q, g, a) for q, g, a in items) for _ in range(n_runs)]  # CORRECT -> 1
    return f1, b1, mean(runs), stdev(runs)                # J: mean +- 1 std over 10 judge runs

**Ваш ответ:**

_Напишите ответ здесь._